In [204]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [205]:
data = pd.read_csv("spotify_streaming_alia.csv", parse_dates=['endTime'])
weather = pd.read_csv("vancouverWeather.csv", parse_dates=['datetime'])

In [206]:
weather = weather[['name', 'datetime', 'temp', 'cloudcover', 'icon']]

In [207]:
data.loc[:, 'datetime'] = data['endTime'].dt.normalize() # removes hours values

In [208]:
merged_data = pd.merge(data, weather, on='datetime', how='inner')
merged_data = merged_data[merged_data['genre'].notnull()]
merged_data['icon'] = merged_data['icon'].astype(str)

**Balancing Data**

In [209]:
counts = merged_data['genre'].value_counts()
counts.mean() # baseline count for number of samples to choose

62.366336633663366

In [210]:
merged_data = pd.merge(merged_data, counts, on='genre', how='inner')

In [211]:
song_data = merged_data[merged_data['count'] >= 60] 

In [212]:
genre_list = song_data['genre'].unique()

In [213]:
genre_list

array(['indie', 'anime', 'seen live', 'rnb', 'dream pop', 'soft pop',
       'pop', 'country', 'female vocalists', 'bedroom pop', 'k-pop',
       'jazz', 'instrumental', 'classical', 'classical piano',
       'Classical', 'j-pop'], dtype=object)

In [214]:
samples = []
for genre in genre_list:
    sample_songs = song_data[song_data['genre'] == genre].sample(60)
    samples.append(sample_songs)
sample_data = pd.concat(samples) 

In [215]:
#sample_data

**Machine Learning**

In [216]:
genre_encoder = preprocessing.LabelEncoder()
icon_encoder = preprocessing.LabelEncoder()

In [217]:
sample_data['genre_code'] = genre_encoder.fit_transform(sample_data['genre'])
sample_data['icon_code'] = icon_encoder.fit_transform(sample_data['icon'])

In [218]:
X = sample_data[['temp', 'cloudcover', 'icon_code']].values
y = sample_data['genre_code'].values

In [219]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y)

In [220]:
# train model using balanced data
model = make_pipeline(
    RandomForestClassifier(n_estimators=100, random_state=42)
)
model.fit(X_train, y_train)

Pipeline(steps=[('randomforestclassifier',
                 RandomForestClassifier(random_state=42))])

In [221]:
model.score(X_valid, y_valid) # really low

0.23137254901960785

**Recommendation**

In [222]:
last_day = pd.to_datetime('2025-02-28') # reference date for the last ten day of listening history
# should be the current day, but data is outdated so it uses the last day of the data instead

In [223]:
# use the whole data
last_ten_days = merged_data[merged_data['datetime'] > (last_day - pd.Timedelta(days=10))] # last ten days

*Sample Weather:* <br>
temp: 12 degree <br>
cloudcover: 83% <br>
icon: clear-day <br>

In [224]:
icon_encoded = icon_encoder.transform(['clear-day'])[0]
sample_temp = np.array([12.0, 83, icon_encoded])
sample_temp = sample_temp.reshape(1,-1)

In [225]:
genre = genre_encoder.inverse_transform(model.predict(sample_temp))[0]

In [226]:
matched_genre = last_ten_days[last_ten_days['genre'] == genre] 
# need to account for genre that wasn't listened to in the last ten days (aka no matches)

Counts the number of listens to determine which song to predict from the last ten days

In [227]:
track_listens = matched_genre['trackName'].value_counts().reset_index().set_axis(['trackName','track_count'], axis=1)

In [228]:
artist_listens = matched_genre['artistName'].value_counts().reset_index().set_axis(['artistName','artist_count'], axis=1)

In [229]:
prediction_data = pd.merge(last_ten_days, track_listens, on='trackName', how='inner')
prediction_data = pd.merge(prediction_data, artist_listens, on='artistName', how='inner')

In [230]:
song_recs = prediction_data.sort_values(by=['track_count', 'artist_count'], ascending=False) # sort by most listened

In [231]:
songs = song_recs['trackName'].unique()

In [232]:
print("Your top three songs based on the weather temperature are:", songs[0:3])

Your top three songs based on the weather temperature are: ['WILDFLOWER' 'BIRDS OF A FEATHER' 'Talking to the Moon']
